# Mel Frequency Cepstral Coefficients (MFCCs)

In [ ]:
import IPython.display
import essentia.standard as ess
import matplotlib.pyplot as plt
import librosa
import numpy
import sklearn

from mirdotcom import mirdotcom

mirdotcom.init()

The [mel frequency cepstral coefficients](https://en.wikipedia.org/wiki/Mel-frequency_cepstrum) (MFCCs) of a signal are a small set of features (usually about 10-20) which concisely describe the overall shape of a spectral envelope. In MIR, it is often used to describe timbre.

Plot the audio signal:

In [ ]:
filename = mirdotcom.get_audio("simple_loop.wav")
x, fs = librosa.load(filename)
librosa.display.waveshow(x, sr=fs)
plt.ylabel("Amplitude")

Play the audio:

In [ ]:
IPython.display.Audio(x, rate=fs)

## Computing MFCCs

[`librosa.feature.mfcc`](https://librosa.org/doc/latest/generated/librosa.feature.mfcc.html#librosa.feature.mfcc) computes MFCCs across an audio signal:


In [ ]:
mfccs = librosa.feature.mfcc(y=x, sr=fs)
print(mfccs.shape)

In this case, `mfcc` computed 20 MFCCs over 130 frames.

The very first MFCC, the 0th coefficient, does not convey information relevant to the overall shape of the spectrum. It only conveys a constant offset, i.e. adding a constant value to the entire spectrum. Therefore, many practitioners will discard the first MFCC when performing classification. For now, we will use the MFCCs as is.

Display the MFCCs:

In [ ]:
librosa.display.specshow(mfccs, sr=fs, x_axis="time")
plt.ylabel("MFCC Coefficients")

### Feature Scaling

Let's scale the MFCCs such that each coefficient dimension has zero mean and unit variance:

In [ ]:
mfccs = sklearn.preprocessing.scale(mfccs, axis=1)
print(mfccs.mean(axis=1))
print(mfccs.var(axis=1))

Display the scaled MFCCs:

In [ ]:
librosa.display.specshow(mfccs, sr=fs, x_axis="time")
plt.ylabel("MFCC Coefficients")

## Alternative Method

We can also use [`essentia.standard.MFCC`](https://essentia.upf.edu/reference/std_MFCC.html) to compute MFCCs across a signal, and we will display them as a "MFCC-gram":

In [ ]:
hamming_window = ess.Windowing(type="hamming")
spectrum = ess.Spectrum()  # we just want the magnitude spectrum
mfcc = ess.MFCC(numberCoefficients=13)
frame_sz = 1024
hop_sz = 500

mfccs = numpy.array(
    [
        mfcc(spectrum(hamming_window(frame)))[1]
        for frame in ess.FrameGenerator(x, frameSize=frame_sz, hopSize=hop_sz)
    ]
)
print(mfccs.shape)

Scale the MFCCs:

In [ ]:
mfccs = sklearn.preprocessing.scale(mfccs)

Plot the MFCCs:

In [ ]:
plt.imshow(mfccs.T, origin="lower", aspect="auto", interpolation="nearest")
plt.ylabel("MFCC Coefficient Index")
plt.xlabel("Frame Index")